# Λ and thresholds at larger code distances (simulated)

**Companion to FT-04 (noise models, thresholds and Λ).** Everything in this notebook is a **classical simulation**, not hardware data.

The noise model is Stim's generated rotated surface-code memory experiment with uniform circuit-level noise at strength $p$. It applies depolarizing noise after every Clifford gate, data depolarization before each round, and bit flips on resets and measurements. Decoding is minimum-weight perfect matching with PyMatching, using the circuit's detector error model.

Real hardware has non-uniform, correlated and leakage errors, so numbers here will not match a particular device. FT-04 uses Google's *measured* $\Lambda=2.14$ for that reason.

Requirements are pinned in `requirements-ft-notebooks.txt`: Stim 1.15.0, PyMatching 2.2.2, NumPy 2.2.6. Runtime is a few minutes on a laptop CPU.

In [1]:
import platform, time
import numpy as np, stim, pymatching
print('stim', stim.__version__, '| pymatching', pymatching.__version__, '| numpy', np.__version__)
print('machine:', platform.platform(), '|', platform.processor() or platform.machine())

stim 1.15.0 | pymatching 2.2.2 | numpy 2.2.6
machine: macOS-27.0.1-arm64-arm-64bit-Mach-O | arm


## One experiment

For distance $d$, run a $Z$-basis memory for $d$ rounds, decode every shot, and count how often the decoded logical observable is wrong. With $P$ the failure probability over $r$ rounds, the per-round logical error $\varepsilon$ solves $P=\tfrac12\bigl(1-(1-2\varepsilon)^r\bigr)$:

$$\varepsilon=\tfrac12\bigl(1-(1-2P)^{1/r}\bigr).$$

In [2]:
def per_round_error(d, p, shots, seed=2026):
    circuit = stim.Circuit.generated(
        'surface_code:rotated_memory_z', distance=d, rounds=d,
        after_clifford_depolarization=p, before_round_data_depolarization=p,
        before_measure_flip_probability=p, after_reset_flip_probability=p)
    matching = pymatching.Matching.from_detector_error_model(circuit.detector_error_model(decompose_errors=True))
    detectors, observables = circuit.compile_detector_sampler(seed=seed).sample(shots, separate_observables=True)
    failures = int((matching.decode_batch(detectors) != observables).any(axis=1).sum())
    P = failures / shots
    eps = 0.5 * (1 - (1 - 2 * P) ** (1 / d))
    # Rough binomial standard error on P, propagated to eps (first order).
    se = (P * (1 - P) / shots) ** 0.5 / (d * max(1 - 2 * P, 1e-9) ** (1 - 1 / d))
    return failures, eps, se

## Sweep distance and physical error

Shots are chosen so that each point sees enough failures for a rough estimate. Points with very few failures are flagged rather than over-interpreted.

In [3]:
distances = [3, 5, 7, 9]
ps = [0.003, 0.005, 0.007, 0.010, 0.013, 0.016]
shots = {0.003: 400_000, 0.005: 200_000, 0.007: 100_000, 0.010: 50_000, 0.013: 40_000, 0.016: 40_000}
results = {}
t0 = time.time()
for p in ps:
    for d in distances:
        results[(d, p)] = per_round_error(d, p, shots[p])
print(f'sweep took {time.time() - t0:.0f} s')
print('per-round logical error eps (failures in brackets; * = fewer than 20 failures)')
print('p       ' + ''.join(f'd={d:<12}' for d in distances))
for p in ps:
    row = ''
    for d in distances:
        f, e, se = results[(d, p)]
        row += f'{e:.2e}({f}){"*" if f < 20 else " "}'.ljust(14)
    print(f'{p:<8}' + row)

sweep took 37 s
per-round logical error eps (failures in brackets; * = fewer than 20 failures)
p       d=3           d=5           d=7           d=9           
0.003   2.20e-03(2629) 6.76e-04(1349) 2.02e-04(564) 6.39e-05(230) 
0.005   5.76e-03(3415) 2.90e-03(2864) 1.47e-03(2042) 7.44e-04(1332) 
0.007   1.06e-02(3099) 7.29e-03(3539) 4.94e-03(3355) 3.62e-03(3169) 
0.01    2.11e-02(3033) 1.78e-02(4150) 1.67e-02(5285) 1.58e-02(6267) 
0.013   3.25e-02(3653) 3.31e-02(5795) 3.59e-02(8130) 3.87e-02(10306) 
0.016   4.56e-02(4990) 5.41e-02(8716) 6.30e-02(12212) 7.34e-02(15213) 


## Suppression factor Λ = ε_d / ε_{d+2}

Λ > 1 means a larger code helps, so the physical error is below this noise model's threshold. Λ < 1 means it is above threshold.

In [4]:
print('p       ' + ''.join(f'Λ{d}/{d+2}'.ljust(12) for d in distances[:-1]))
for p in ps:
    row = ''
    for d in distances[:-1]:
        e1, e2 = results[(d, p)][1], results[(d + 2, p)][1]
        row += (f'{e1 / e2:.2f}' if e2 > 0 else 'n/a').ljust(12)
    print(f'{p:<8}' + row)

p       Λ3/5        Λ5/7        Λ7/9        
0.003   3.25        3.35        3.15        
0.005   1.99        1.97        1.98        
0.007   1.45        1.48        1.36        
0.01    1.18        1.07        1.06        
0.013   0.98        0.92        0.93        
0.016   0.84        0.86        0.86        


## Reading the result

These numbers come from fixed random seeds, so re-running the notebook reproduces them exactly.

- **Λ grows as $p$ falls.** Λ is about 3.2 at $p=0.3\%$, about 2.0 at $0.5\%$, about 1.4 at $0.7\%$, and about 1.1 at $1\%$. This is the $\Lambda\approx p_{\rm th}/p$ trend that FT-04's rule of thumb describes.
- **The threshold.** In this uniform model, Λ reaches 1 somewhere around $p\approx1\%$ to $1.3\%$; read the exact crossing off the two highest-$p$ rows. That is higher than the 0.57% Fowler et al. quote for their own circuit-level model (FT-04), because different noise models and decoders give different thresholds. Neither number describes a particular device.
- **Google's measured Λ = 2.14** lands, in this simulation, near a uniform $p\approx0.5\%$. There the simulated distance-7 error is 0.147% per round, close to Google's measured 0.143%. That closeness is a coincidence of scale, not a calibration: hardware noise is not uniform.
- **Limits.** These are $Z$-basis memories with uncorrelated matching and no leakage, correlated bursts or logical gates. Points marked * have too few failures to trust.